# Weather Pattern Footprints and High-Risk Pattern (HRP) Analysis

**Merged from:** `WP_footprints_incident_days.ipynb` and `HRP_analysis.ipynb`

This notebook has two parts:

**Part 1 — Per-day footprint generation**  
For each day in a weather-pattern (WP) subset dictionary, computes a gridded
classification array encoding whether each grid cell is cold-only, wet-only,
cold-wet (CW), or cold-dry (CD). Incident locations are overlaid. Results are
stacked into per-WP 3-D arrays and saved to disk.

**Part 2 — Modal (dominant) footprint maps**  
Loads the stacked per-WP arrays and computes the modal (most frequently
occurring) classification at each grid cell across all days in a WP subset.
Produces a 6×5 overview map of all WPs, then a focused figure of the
high-risk patterns (HRPs) most associated with compound cold events.

**Inputs**
| File/directory | Description |
|---|---|
| `data/arrays/{month}_min_temp_all.npy` | Monthly minimum temperature arrays |
| `data/arrays/{month}_precip_all.npy` | Monthly precipitation arrays |
| `data/percentiles/minT15P_30yr_filtered.npy` | 15th-percentile temperature threshold |
| `data/percentiles/perc15P_30yr_filtered.npy` | 15th-percentile precipitation threshold |
| `data/percentiles/perc85P_30yr_filtered.npy` | 85th-percentile precipitation threshold |
| `data/all_incident_data.csv` | Incident records with Lat, Long, Date columns |
| `data/scotland_rail.shp` | Rail network shapefile |
| `data/wp_subsetA_dictionaries/wp_{N}_subsetA_data.pkl` | Per-WP day metadata (one per WP) |
| HadUK-Grid sample `.nc` | Used to read lat/lon coordinate grid |

**Outputs**
| Directory / file | Description |
|---|---|
| `output/combined_wp_subsetA_output_arrays/` | Stacked per-WP classification arrays |
| `output/figures/WP_dominant_WT_subsetA_days.png` | All-WP modal footprint overview |
| `output/figures/hrp_subset_days.png` | Focused HRP comparison figure |

**Configuration:** Set the paths in the cell below before running.


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from netCDF4 import Dataset
from pathlib import Path
from scipy.stats import mode

# ── User configuration ──────────────────────────────────────────────────────
ARRAYS_DIR      = "data/arrays"
PERCENTILES_DIR = "data/percentiles"
INCIDENT_CSV    = "data/all_incident_data.csv"
RAIL_SHAPEFILE  = "data/scotland_rail.shp"
WP_DICT_DIR     = Path("data/wp_subsetA_dictionaries")
OUTPUT_ARRAYS   = Path("output/combined_wp_subsetA_output_arrays")
FIG_DIR         = Path("output/figures")
HADUK_SAMPLE    = ("/path/to/HadUK-Grid/5km/tasmin/day/latest/"
                   "tasmin_hadukgrid_uk_5km_day_19600101-19600131.nc")

# Map extent [lon_min, lon_max, lat_min, lat_max]
MAP_EXTENT = [-8.0, -0.8, 54.5, 60.9]

# WPs to skip (e.g. those absent from the classification scheme)
WPS_TO_SKIP = {3}

# High-risk patterns to highlight in the focused figure
HRP_NUMBERS = [19, 27, 28]

# Fill value threshold for percentile arrays
FILL_VALUE = 500

OUTPUT_ARRAYS.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration complete.")


## 1. Load shared data (coordinates, incidents, rail network)

In [ ]:
# Load lat/lon from a sample HadUK-Grid file
with Dataset(HADUK_SAMPLE, mode='r') as ds:
    lat = ds.variables['latitude'][:]
    lon = ds.variables['longitude'][:]

# Load percentile threshold arrays and mask fill values
perc_minT_15 = np.load(os.path.join(PERCENTILES_DIR, "minT15P_30yr_filtered.npy"))
perc_prec_15 = np.load(os.path.join(PERCENTILES_DIR, "perc15P_30yr_filtered.npy"))
perc_prec_85 = np.load(os.path.join(PERCENTILES_DIR, "perc85P_30yr_filtered.npy"))
for arr in [perc_minT_15, perc_prec_15, perc_prec_85]:
    arr[arr > FILL_VALUE] = np.nan

# Load incident dataset
df_incidents = pd.read_csv(INCIDENT_CSV)
gdf_incidents = gpd.GeoDataFrame(
    df_incidents,
    geometry=gpd.points_from_xy(df_incidents['Long'], df_incidents['Lat']),
    crs="EPSG:4326"
)

# Load rail shapefile
scot_rail = gpd.read_file(RAIL_SHAPEFILE)
if scot_rail.crs != "EPSG:4326":
    scot_rail = scot_rail.to_crs("EPSG:4326")

print(f"Loaded {len(df_incidents):,} incident records.")


## Part 1 — Per-day footprint generation

### 2. Core footprint function

In [ ]:
def make_footprint(input_month, year_index, day_index, date, WP, ax):
    """
    Compute and plot the compound-event classification for a single day,
    then overlay incident locations.

    Classification encoding
    -----------------------
    0 — Neither cold nor wet/dry threshold met
    1 — Cold only (temperature threshold met)
    2 — Wet only  (85th-percentile precipitation threshold met)
    3 — Cold-Wet  (both cold and wet thresholds met)
    4 — Cold-Dry  (both cold and dry thresholds met)

    Parameters
    ----------
    input_month : str   e.g. 'jan'
    year_index  : int   Years since 1960
    day_index   : int   Zero-based day-of-month index
    date        : str   ISO date string 'YYYY-MM-DD'
    WP          : str or int   Weather-pattern number (used for labelling only)
    ax          : matplotlib Axes with a Cartopy projection

    Returns
    -------
    output_array      : ndarray (lat, lon)   Classification values 0–4
    mesh              : QuadMesh             pcolormesh object (for colorbars)
    incident_reason_counts : DataFrame       Incident reason counts for this date
    """
    # Load monthly arrays
    temp_all   = np.load(os.path.join(ARRAYS_DIR, f"{input_month}_min_temp_all.npy"))
    precip_all = np.load(os.path.join(ARRAYS_DIR, f"{input_month}_precip_all.npy"))

    # Extract single day
    temp_day   = temp_all[year_index,   day_index, :, :]
    precip_day = precip_all[year_index, day_index, :, :]

    # Threshold flags
    temp_met      = temp_day   <= perc_minT_15
    precip_met_CW = precip_day >= perc_prec_85
    precip_met_CD = precip_day <= perc_prec_15

    # Encode classification
    output_array = np.zeros(temp_day.shape)
    output_array[temp_met]                     = 1
    output_array[precip_met_CW]                = 2
    output_array[temp_met & precip_met_CW]     = 3
    output_array[temp_met & precip_met_CD]     = 4

    # Filter incidents to this date and aggregate by location
    filtered = gdf_incidents[gdf_incidents['Date'] == str(date)]
    incident_counts = filtered.groupby(['Long', 'Lat']).size().reset_index(name='count')
    incident_reason_counts = (filtered
        .groupby(['Incident Reason Name', 'PfPI Minutes'])
        .size().reset_index(name='Count'))

    # Plot
    cmap_picked = mpl.colors.ListedColormap(['white', 'gold', 'dodgerblue', 'seagreen', 'magenta'])
    bounds = np.array([0, 1, 2, 3, 4, 5])
    norm   = mpl.colors.BoundaryNorm(bounds, cmap_picked.N)

    mesh = ax.pcolormesh(lon, lat, output_array, shading='auto',
                         cmap=cmap_picked, norm=norm, transform=ccrs.PlateCarree())
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.scatter(incident_counts['Long'], incident_counts['Lat'],
               s=incident_counts['count'] * 10, color='black', alpha=0.6,
               transform=ccrs.PlateCarree())
    ax.set_title(f"{date}  WP{WP}")

    return output_array, mesh, incident_reason_counts


### 3. Loop over all WPs — generate footprints and save stacked arrays

In [ ]:
for wp_num in range(1, 31):
    if wp_num in WPS_TO_SKIP:
        print(f"Skipping WP {wp_num} (excluded).")
        continue

    dict_path = WP_DICT_DIR / f"wp_{wp_num}_subsetA_data.pkl"
    if not dict_path.is_file():
        print(f"No data file for WP {wp_num} — skipping.")
        continue

    with open(dict_path, 'rb') as f:
        wp_dict = pickle.load(f)

    num_days = len(wp_dict)
    if num_days == 0:
        print(f"WP {wp_num}: 0 days — skipping.")
        continue

    print(f"Processing WP {wp_num} ({num_days} days)")

    # Build subplot grid
    if num_days == 1:
        fig, axs = plt.subplots(figsize=(8, 8),
                                subplot_kw={'projection': ccrs.PlateCarree()})
        axs = [axs]
    else:
        rows = int(np.ceil(np.sqrt(num_days)))
        cols = int(np.ceil(num_days / rows))
        fig, axs = plt.subplots(rows, cols, figsize=(15, 15),
                                subplot_kw={'projection': ccrs.PlateCarree()})
        axs = axs.ravel()

    combined_output = None
    for idx, (date, info) in enumerate(wp_dict.items()):
        if idx >= len(axs):
            break
        output_array, mesh, _ = make_footprint(
            input_month=info['input_month'],
            year_index=info['year_index'],
            day_index=info['day_index'],
            date=info['date'],
            WP=info['WP'],
            ax=axs[idx]
        )
        if combined_output is None:
            combined_output = np.zeros((num_days, *output_array.shape),
                                       dtype=output_array.dtype)
        combined_output[idx] = output_array

    # Hide any unused axes
    for i in range(num_days, len(axs)):
        fig.delaxes(axs[i])

    plt.tight_layout()
    fig.suptitle(f"Weather Pattern {wp_num} — Footprints", fontsize=16)
    plt.subplots_adjust(top=0.95)
    # fig.savefig(FIG_DIR / f"WP_{wp_num}_footprints.png", dpi=300)
    plt.show()

    # Save stacked array
    out_path = OUTPUT_ARRAYS / f"combined_output_wp_{wp_num}_subsetA.npy"
    np.save(out_path, combined_output)
    print(f"  Saved stacked array to {out_path}")


## Part 2 — Modal footprint maps

### 4. All-WP overview (6 × 5 grid)

Computes the modal (most frequently occurring) classification value at each
grid cell across all days in each WP's subset, and plots all WPs in a single
overview figure.


In [ ]:
cmap_all = mpl.colors.ListedColormap(['white', 'gold', 'dodgerblue', 'seagreen', 'magenta'])
bounds_all = np.array([0, 1, 2, 3, 4, 5])
norm_all   = mpl.colors.BoundaryNorm(bounds_all, cmap_all.N)

fig, axs = plt.subplots(6, 5, figsize=(20, 24),
                         subplot_kw={'projection': ccrs.PlateCarree()})
fig.subplots_adjust(hspace=0.3, wspace=0.3)

wp_counter = 0
mesh = None

for wp_num in range(1, 31):
    if wp_num in WPS_TO_SKIP:
        print(f"Skipping WP {wp_num}.")
        continue
    if wp_counter >= len(axs.flat):
        break

    ax = axs.flat[wp_counter]
    wp_counter += 1

    arr_path = OUTPUT_ARRAYS / f"combined_output_wp_{wp_num}_subsetA.npy"
    if not arr_path.exists():
        print(f"No stacked array for WP {wp_num} — skipping.")
        ax.axis('off')
        continue

    wp_combined = np.load(arr_path)
    mode_result = mode(wp_combined, axis=0)
    mode_array  = mode_result.mode.squeeze()

    mesh = ax.pcolormesh(lon, lat, mode_array, shading='auto',
                         cmap=cmap_all, norm=norm_all,
                         transform=ccrs.PlateCarree())
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_title(f"WP {wp_num}", fontsize=10)

# Remove unused subplots
for i in range(wp_counter, len(axs.flat)):
    fig.delaxes(axs.flat[i])

if mesh is not None:
    cbar = plt.colorbar(mesh, ax=axs, ticks=[0, 1, 2, 3, 4],
                        orientation='vertical', fraction=0.025, pad=0.04, shrink=0.8)
    cbar.ax.set_yticklabels(['Neither', 'Temp Only', 'Precip Only', 'Cold-Wet', 'Cold-Dry'])

plt.savefig(FIG_DIR / 'WP_dominant_WT_subsetA_days.png', dpi=300, bbox_inches='tight')
plt.show()


### 5. Focused figure — high-risk patterns (HRPs) only

In [ ]:
# Two-colour map: only Cold-Wet and Cold-Dry shown; all other values are white
cmap_hrp = mpl.colors.ListedColormap(['white', 'seagreen', 'magenta'])
bounds_hrp = np.array([0, 3, 4, 5])
norm_hrp   = mpl.colors.BoundaryNorm(bounds_hrp, cmap_hrp.N)

fig, axs = plt.subplots(1, len(HRP_NUMBERS), figsize=(8, 14),
                         subplot_kw={'projection': ccrs.PlateCarree()})
axs = np.atleast_1d(axs).flatten()
mesh_hrp = None

for ax, wp_num in zip(axs, HRP_NUMBERS):
    arr_path = OUTPUT_ARRAYS / f"combined_output_wp_{wp_num}_subsetA.npy"
    if not arr_path.exists():
        print(f"No data for WP {wp_num}")
        ax.set_title(f"WP {wp_num} (No Data)")
        ax.axis('off')
        continue

    wp_combined = np.load(arr_path)
    mode_result = mode(wp_combined, axis=0)
    mode_array  = mode_result.mode.squeeze()

    mesh_hrp = ax.pcolormesh(lon, lat, mode_array, shading='auto',
                              cmap=cmap_hrp, norm=norm_hrp,
                              transform=ccrs.PlateCarree())
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.75)
    scot_rail.plot(ax=ax, color='black', linestyle='dotted', linewidth=1.2,
                   transform=ccrs.PlateCarree(), alpha=0.6)
    ax.set_title(f"WP {wp_num}", fontsize=10)

# Remove unused axes
for i in range(len(HRP_NUMBERS), len(axs)):
    fig.delaxes(axs[i])

if mesh_hrp is not None:
    cbar = plt.colorbar(mesh_hrp, ax=axs, ticks=[0, 3, 4],
                        orientation='vertical', fraction=0.025, pad=0.04, shrink=0.8)
    cbar.ax.set_yticklabels(['Neither', 'Cold-Wet', 'Cold-Dry'])

plt.savefig(FIG_DIR / 'hrp_subset_days.png', dpi=300, bbox_inches='tight')
plt.show()
